In [1]:
!pip install ultralytics pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [2]:
import random
import sys
import numpy as np
import torch

from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION
if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent
                         for path in INPUT_ROOT.rglob("annotations_coco.json")
                         if path.parent.name == "train"})
    if len(data_roots) != 1:
        raise FileNotFoundError(
            "Attach the extracted competition data, or set DIRECTORY to its input folder."
        )
    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "torch_model.py",
    "yolo26_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("yolo26_model.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file() for filename in required_scripts)
})
scripts_parent = str(script_dirs[0].parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations, save_fold_yaml, load_yaml
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.experiment_runner import run_yolo_cv
from scripts.experiment_results import model_training_results_yolo_cv

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
BATCH_SIZE = 8
FRAME_STEP = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=6.29s)
creating index...
index created!


In [5]:
cv_fold_loaders, fold_yaml_paths = {}, {}

yolo_dir = (DIRECTORY / "train").resolve()
class_names = load_yaml(yolo_dir / "data.yaml")["names"]

fold_output_dir = PROJECT_ROOT / "yolo_cv"
folds_dir = fold_output_dir / "folds"
folds_dir.mkdir(parents=True, exist_ok=True)

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    for split, split_frames in [("train", training_frames), ("val", validation_frames),]:
        image_paths = [str((DIRECTORY / frame["split"] / frame["file_name"]).resolve())
            for _, frame in split_frames.iterrows()
        ]

        (folds_dir / f"fold_{fold}_{split}.txt").write_text("\n".join(image_paths) + "\n", encoding="utf-8")

    fold_yaml_paths[fold] = save_fold_yaml(dataset_dir=yolo_dir, output_dir=fold_output_dir, fold=fold, class_names=class_names)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(training_frames)} training frames, {len(validation_frames)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


In [6]:
cv_kwargs = {"cv_fold_loaders": cv_fold_loaders, "fold_yaml_paths": fold_yaml_paths,
             "annotations_path": INFO_TRAIN, "device": device, "seed": SEED,
             "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": 1e-4, 
             "weight_decay": 1e-4, "optimizer": "Adam", "validate_each_epoch": False}
fold_dirs = {fold: f"fold_{fold}" for fold in range(FOLDS)}

In [7]:
baseline_histories, baseline_results = run_yolo_cv(**cv_kwargs, output_dir=PROJECT_ROOT / "outputs" / "yolo26_baseline_v2",
                                                   imgsz=640, box_weight=7.5)

loading annotations into memory...
Done (t=5.29s)
creating index...
index created!
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosa

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/5       1.6G      2.297      1.967   0.003931         62        640: 100% ━━━━━━━━━━━━ 139/139 7.9it/s 17.7s
Fold 0 | Epoch 2/5 | loss=4.2681 | learning_rate=0.000053
['Fold 0', 'Epoch 2/5', 'loss=4.2681', 'learning_rate=0.000053']

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        3/5       1.6G      2.228      1.883   0.003752        144        640: 100% ━━━━━━━━━━━━ 139/139 7.6it/s 18.4s
Fold 0 | Epoch 3/5 | loss=4.1149 | learning_rate=0.000060
['Fold 0', 'Epoch 3/5', 'loss=4.1149', 'learning_rate=0.000060']

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        4/5       1.6G      2.213      1.847   0.003722        124        640: 100% ━━━━━━━━━━━━ 139/139 7.7it/s 18.1s
Fold 0 | Epoch 4/5 | loss=4.0634 | learning_rate=0.000041
['Fold 0', 'Epoch 4/5', 'loss=4.0634', 'learning_rate=0.000041']

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        5/5       1.6G      2.198    

In [8]:
baseline_results_wrap = model_training_results_yolo_cv(PROJECT_ROOT, filename="yolo26_baseline_v2")

Final validation results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,4.028758,0.000021,0.070883,0.143081,0.011915
1,1,5,4.257132,0.000021,0.078074,0.153604,0.022963
2,2,5,4.157253,0.000021,0.062448,0.126201,0.015255



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.070468,0.140962,0.016711
std,0.007821,0.013824,0.005666


In [9]:
resolution_histories, resolution_results = run_yolo_cv(**cv_kwargs, output_dir=PROJECT_ROOT / "outputs" / "yolo26_res1024", imgsz=1024, box_weight=7.5)

loading annotations into memory...
Done (t=5.11s)
creating index...
index created!
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mos

In [10]:
resolution_results_wrap =  model_training_results_yolo_cv(PROJECT_ROOT, "yolo26_res1024")

Final validation results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,3.648657,0.000021,0.114911,0.220943,0.029350
1,1,5,3.874518,0.000021,0.125663,0.236429,0.040508
2,2,5,3.793124,0.000021,0.100303,0.194719,0.032385



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.113626,0.217364,0.034081
std,0.012729,0.021084,0.005769


In [11]:
box_histories, box_results = run_yolo_cv(**cv_kwargs, output_dir=PROJECT_ROOT / "outputs" / "yolo26_box", imgsz=1024, box_weight=10.0)

loading annotations into memory...
Done (t=5.10s)
creating index...
index created!
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mo

In [12]:
box_results_wrap = model_training_results_yolo_cv(PROJECT_ROOT, "yolo26_box")

Final validation results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,4.300518,0.000021,0.110873,0.210263,0.030204
1,1,5,4.551395,0.000021,0.126686,0.237058,0.039484
2,2,5,4.465584,0.000021,0.104055,0.199413,0.033606



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.113871,0.215578,0.034431
std,0.011609,0.019377,0.004695
